# RIDE Package — Notebook Test

End-to-end test of every wrang feature used as a Python library inside a Jupyter notebook.

**Setup:** `pip install -e .` from the repo root, then run all cells.

In [1]:
import polars as pl
import json
from pathlib import Path

# Core imports
from wrang.core.loader      import FastDataLoader, DataSaver, load_data, save_data
from wrang.core.inspector   import DataInspector, inspect_data
from wrang.core.explorer    import DataExplorer, explore_data
from wrang.core.cleaner     import DataCleaner, quick_clean
from wrang.core.transformer import DataTransformer, quick_transform
from wrang.core.validator   import DataSchema, ColumnSchema, DataValidator, infer_schema, validate_data
from wrang.config import (
    get_config, update_config, reset_config,
    ImputationStrategy, ScalingMethod, EncodingMethod,
)

print('All imports OK — wrang is ready')

All imports OK — wrang is ready


## 1. Sample Dataset

In [2]:
df = pl.DataFrame({
    'id':     [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'name':   ['Alice', 'Bob', 'Carol', 'Dave', 'Eve',
               'Frank', 'Grace', None, 'Henry', 'Iris'],
    'age':    [25, 30, 22, 35, 28, 45, 31, None, 27, 29],
    'salary': [55000.0, 72000.0, 48000.0, 95000.0, 61000.0,
               120000.0, 68000.0, 53000.0, 59000.0, None],
    'dept':   ['eng', 'eng', 'hr', 'eng', 'hr',
               'eng', 'hr', 'hr', 'eng', 'hr'],
    'score':  [88.5, 92.0, 76.3, 95.0, 82.1,
               97.5, 78.9, 65.0, 85.3, 90.0],
    'active': [True, True, True, True, False,
               True, True, False, True, True],
})

print(f'Shape: {df.shape}')
df

Shape: (10, 7)


id,name,age,salary,dept,score,active
i64,str,i64,f64,str,f64,bool
1,"""Alice""",25,55000.0,"""eng""",88.5,true
2,"""Bob""",30,72000.0,"""eng""",92.0,true
3,"""Carol""",22,48000.0,"""hr""",76.3,true
4,"""Dave""",35,95000.0,"""eng""",95.0,true
5,"""Eve""",28,61000.0,"""hr""",82.1,false
6,"""Frank""",45,120000.0,"""eng""",97.5,true
7,"""Grace""",31,68000.0,"""hr""",78.9,true
8,null,null,53000.0,"""hr""",65.0,false
9,"""Henry""",27,59000.0,"""eng""",85.3,true


## 2. Loader

In [4]:
tmp = Path('/tmp/ride_notebook_test')
tmp.mkdir(exist_ok=True)

saver  = DataSaver()
loader = FastDataLoader()

# Save in multiple formats
saver.save(df, tmp / 'employees.csv')
saver.save(df, tmp / 'employees.parquet')
print('Saved CSV and Parquet')

# Reload
df_csv     = loader.load(tmp / 'employees.csv')
df_parquet = loader.load(tmp / 'employees.parquet')
print(f'CSV:     {df_csv.shape}')
print(f'Parquet: {df_parquet.shape}')

✅ Successfully saved 10 rows to employees.csv

✅ Successfully saved 10 rows to employees.parquet

Saved CSV and Parquet


✅ Successfully loaded 10 rows × 7 columns

✅ Successfully loaded 10 rows × 7 columns

CSV:     (10, 7)
Parquet: (10, 7)


In [5]:
# Peek — fast preview without loading entire file
peek = loader.peek(tmp / 'employees.csv', n_rows=3)
print(f'Peek shape: {peek.shape}')
peek

Peek shape: (3, 7)


id,name,age,salary,dept,score,active
i64,str,i64,f64,str,f64,bool
1,"""Alice""",25,55000.0,"""eng""",88.5,true
2,"""Bob""",30,72000.0,"""eng""",92.0,true
3,"""Carol""",22,48000.0,"""hr""",76.3,true


In [6]:
# Streaming — iterate in chunks
chunks = list(loader.stream_chunks(tmp / 'employees.csv', chunk_size=3))
total  = sum(len(c) for c in chunks)
print(f'Streaming: {len(chunks)} chunk(s), {total} total rows')

# Real-world dataset (if available)
titanic_path = Path('datasets/titanic.csv')
if titanic_path.exists():
    df_titanic = load_data(titanic_path)
    print(f'Titanic: {df_titanic.shape}')
    display(df_titanic.head(3))

Streaming: 1 chunk(s), 10 total rows


✅ Successfully loaded 891 rows × 12 columns

Titanic: (891, 12)


PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S"""
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S"""


## 3. Inspector

In [7]:
inspector = inspect_data(df)

info = inspector.get_basic_info()
print('Basic info:')
for k, v in info.items():
    print(f'  {k}: {v}')

Basic info:
  n_rows: 10
  n_columns: 7
  memory_usage: {'total_memory_mb': 0.0003719329833984375, 'total_memory_human': '0.4 KB', 'column_memory': {'id': 7.62939453125e-05, 'name': 3.719329833984375e-05, 'age': 7.82012939453125e-05, 'salary': 7.82012939453125e-05, 'dept': 2.384185791015625e-05, 'score': 7.62939453125e-05, 'active': 1.9073486328125e-06}, 'largest_columns': [('age', 7.82012939453125e-05), ('salary', 7.82012939453125e-05), ('id', 7.62939453125e-05), ('score', 7.62939453125e-05), ('name', 3.719329833984375e-05)]}
  column_types: {'id': 'Int64', 'name': 'String', 'age': 'Int64', 'salary': 'Float64', 'dept': 'String', 'score': 'Float64', 'active': 'Boolean'}
  missing_values_total: 3
  duplicate_rows: 0
  numeric_columns: 4
  categorical_columns: 2
  datetime_columns: 0


In [8]:
# Full overview panel (Rich output)
inspector.display_overview()

╭────────────────────────────────────────────── 📊 Dataset Overview ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Dataset Dimensions:                                                                                            │
│  • Rows: 10                                                                                                     │
│  • Columns: 7                                                                                                   │
│  • Memory Usage: 0.4 KB                                                                                         │
│                                                                                                                 │
│  Column Types:                                                                                                  │
│  • Numeric: 4                                                                                                   │
│  • Categorical: 2                                                                                               │
│  • DateTime: 0                                                                                                  │
│                                                                                                                 │
│  Data Quality:                                                                                                  │
│  • Missing Values: 3 cells                                                                                      │
│  • Duplicate Rows: 0                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                                 📋 Column Details                                                 
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column               ┃ Type         ┃ Missing  ┃ Missing %  ┃ Unique   ┃ Unique %   ┃ Sample Values             ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ id                   │ Int64        │ 0        │ 0%         │ 10       │ 100.0%     │ 1, 2, 3                   │
│ name                 │ String       │ 1        │ 10.0%      │ 10       │ 100.0%     │ Alice, Bob, Carol         │
│ age                  │ Int64        │ 1        │ 10.0%      │ 10       │ 100.0%     │ 25, 30, 22                │
│ salary               │ Float64      │ 1        │ 10.0%      │ 10       │ 100.0%     │ 55000.0, 72000.0, 48000.0 │
│ dept                 │ String       │ 0        │ 0%         │ 2        │ 20.0%      │ eng, eng, hr              │
│ score                │ Float64      │ 0        │ 0%         │ 10       │ 100.0%     │ 88.5, 92.0, 76.3          │
│ active               │ Boolean      │ 0        │ 0%         │ 2        │ 20.0%      │ True, True, True          │
└──────────────────────┴──────────────┴──────────┴────────────┴──────────┴────────────┴───────────────────────────┘

╭──── 🚫 Missing Values ────╮             ╭────── 🔧 Data Types ──────╮         ╭────── 🔄 Duplicates ──────╮      
│ Missing Values Summary    │             │ Data Types Distribution   │         │ Duplicate Analysis        │      
│                           │             │                           │         │                           │      
│ • Columns with missing: 3 │             │ • Numeric: 4              │         │ • Duplicate rows: 0       │      
│ • Total missing cells: 3  │             │ • Categorical: 2          │         │ • Duplicate %: 0.0%       │      
│ • Overall missing %: 4.3% │             │ • DateTime: 0             │         │ • Unique rows: 10         │      
│                           │             │                           │         │                           │      
│ Worst Columns:            │             │ Type Details:             │         │ Status:                   │      
│ • name: 10.0%             │             │ • Int64: 2                │         │ ✅ No duplicates found    │      
│ • age: 10.0%              │             │ • String: 2               │         ╰───────────────────────────╯      
│ • salary: 10.0%           │             │ • Float64: 2              │                                            
╰───────────────────────────╯             │ • Boolean: 1              │                                            
                                          ╰───────────────────────────╯

In [9]:
# Column profiles — programmatic access
profiles = inspector.get_column_profiles()
print('Column profiles:')
for col, p in profiles.items():
    print(f"  {col:10} dtype={p['dtype']:12}  missing={p['missing_percentage']:.1f}%  unique={p['unique_count']}")

Column profiles:
  id         dtype=Int64         missing=0.0%  unique=10
  name       dtype=String        missing=10.0%  unique=10
  age        dtype=Int64         missing=10.0%  unique=10
  salary     dtype=Float64       missing=10.0%  unique=10
  dept       dtype=String        missing=0.0%  unique=2
  score      dtype=Float64       missing=0.0%  unique=10
  active     dtype=Boolean       missing=0.0%  unique=2


In [10]:
# Data quality report
inspector.display_data_quality()

╭──── 🚫 Missing Values ────╮             ╭────── 🔧 Data Types ──────╮         ╭────── 🔄 Duplicates ──────╮      
│ Missing Values Summary    │             │ Data Types Distribution   │         │ Duplicate Analysis        │      
│                           │             │                           │         │                           │      
│ • Columns with missing: 3 │             │ • Numeric: 4              │         │ • Duplicate rows: 0       │      
│ • Total missing cells: 3  │             │ • Categorical: 2          │         │ • Duplicate %: 0.0%       │      
│ • Overall missing %: 4.3% │             │ • DateTime: 0             │         │ • Unique rows: 10         │      
│                           │             │                           │         │                           │      
│ Worst Columns:            │             │ Type Details:             │         │ Status:                   │      
│ • name: 10.0%             │             │ • Int64: 2                │         │ ✅ No duplicates found    │      
│ • age: 10.0%              │             │ • String: 2               │         ╰───────────────────────────╯      
│ • salary: 10.0%           │             │ • Float64: 2              │                                            
╰───────────────────────────╯             │ • Boolean: 1              │                                            
                                          ╰───────────────────────────╯

In [11]:
# Statistical summary
inspector.display_statistical_summary()

                                                 📈 Statistical Summary                                            
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━
┃ Column          ┃ Count    ┃ Mean       ┃ Std        ┃ Min        ┃ 25%        ┃ 50%        ┃ 75%        ┃ Max   
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━
│ id              │ 10       │ 5.500      │ 3.028      │ 1.000      │ 3.000      │ 5.500      │ 8.000      │ 10.000
│ age             │ 9        │ 30.222     │ 6.648      │ 22.000     │ 27.000     │ 29.000     │ 31.000     │ 45.000
│ salary          │ 9        │ 70111.111  │ 23272.540  │ 48000.000  │ 55000.000  │ 61000.000  │ 72000.000  │ 120000
│ score           │ 10       │ 85.060     │ 9.797      │ 65.000     │ 78.900     │ 86.900     │ 92.000     │ 97.500
└─────────────────┴──────────┴────────────┴────────────┴────────────┴────────────┴────────────┴────────────┴───────

In [12]:
# Memory usage
mem = inspector.get_memory_usage()
print('Memory usage keys:', list(mem.keys())[:6])

# Detect potential issues
issues = inspector.detect_potential_issues()
print(f'\nDetected {len(issues)} issue(s):')
for i in issues:
    print(f"  [{i.get('severity','?').upper():6}] {i.get('type','?'):25} col={i.get('column','?')}")

Memory usage keys: ['total_memory_mb', 'total_memory_human', 'column_memory', 'largest_columns']

Detected 0 issue(s):


## 4. Explorer

In [13]:
num_df   = df.select(['age', 'salary', 'score']).drop_nulls()
explorer = explore_data(num_df)

# Correlations
corr   = explorer.analyze_correlations(min_correlation=0.0)
pairs  = corr['correlations']
print(f'Correlation pairs: {len(pairs)}')
for p in pairs:
    print(f"  {p['column1']:8} × {p['column2']:8}  r={p['correlation']:.3f}  {p['strength']}")

                         Correlation Analysis (Pearson)                         
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Column 1        ┃ Column 2        ┃ Correlation  ┃ Strength     ┃ Direction  ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ age             │ salary          │ 0.9843       │ Very Strong  │ Positive   │
│ salary          │ score           │ 0.8037       │ Strong       │ Positive   │
│ age             │ score           │ 0.7453       │ Strong       │ Positive   │
└─────────────────┴─────────────────┴──────────────┴──────────────┴────────────┘

Correlation pairs: 3
  age      × salary    r=0.984  Very Strong
  salary   × score     r=0.804  Strong
  age      × score     r=0.745  Strong


In [14]:
# Distributions
dist = explorer.analyze_distributions()['distributions']
print('Distribution stats:')
for col, s in dist.items():
    print(f"  {col:8}  mean={s['mean']:.1f}  std={s['std']:.1f}  skew={s['skewness']:.2f}  kurt={s['kurtosis']:.2f}")

📊 Distribution Analysis

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ age                                                                                                             │
│ • Count: 8                                                                                                      │
│ • Mean: 30.375, Median: 29.000                                                                                  │
│ • Std: 6.632, Range: 23.000                                                                                     │
│ • Skewness: 1.042, Kurtosis: 0.368                                                                              │
│ • Distribution: highly skewed                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ salary                                                                                                          │
│ • Count: 8                                                                                                      │
│ • Mean: 72250.000, Median: 64500.000                                                                            │
│ • Std: 22370.460, Range: 72000.000                                                                              │
│ • Skewness: 1.088, Kurtosis: -0.031                                                                             │
│ • Distribution: highly skewed                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ score                                                                                                           │
│ • Count: 8                                                                                                      │
│ • Mean: 86.950, Median: 86.900                                                                                  │
│ • Std: 7.146, Range: 21.200                                                                                     │
│ • Skewness: -0.007, Kurtosis: -1.328                                                                            │
│ • Distribution: approximately symmetric                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Distribution stats:
  age       mean=30.4  std=6.6  skew=1.04  kurt=0.37
  salary    mean=72250.0  std=22370.5  skew=1.09  kurt=-0.03
  score     mean=86.9  std=7.1  skew=-0.01  kurt=-1.33


In [15]:
# Outlier detection
outliers = explorer.detect_outliers(method='iqr')['outliers']
print('Outlier detection (IQR):')
for col, o in outliers.items():
    print(f"  {col:8}  count={o['outlier_count']}  bounds={o['bounds']}")

🎯 Outlier Detection (IQR)

                              Outlier Summary                              
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Column          ┃ Total Values ┃ Outliers   ┃ Outlier %  ┃ Method       ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ age             │ 8            │ 1          │ 12.5%      │ IQR          │
│ salary          │ 8            │ 1          │ 12.5%      │ IQR          │
│ score           │ 8            │ 0          │ 0.0%       │ IQR          │
└─────────────────┴──────────────┴────────────┴────────────┴──────────────┘

Outlier detection (IQR):
  age       count=1  bounds=40.25
  salary    count=1  bounds=107375.0
  score     count=0  bounds=109.92500000000001


In [16]:
# Normality tests
norm = explorer.test_normality()['normality_tests']
print('Shapiro-Wilk tests:')
for col, r in norm.items():
    print(f"  {col:8}  p={r['p_value']:.4f}  normal={r['is_normal']}")

                           Normality Tests (Shapiro-Wilk)                            
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column          ┃ Statistic    ┃ P-value      ┃ Normal?    ┃ Interpretation       ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ age             │ 0.9099       │ 0.3530       │ Yes        │ Likely normal        │
│ salary          │ 0.8662       │ 0.1383       │ Yes        │ Likely normal        │
│ score           │ 0.9631       │ 0.8387       │ Yes        │ Likely normal        │
└─────────────────┴──────────────┴──────────────┴────────────┴──────────────────────┘

Shapiro-Wilk tests:
  age       p=0.3530  normal=True
  salary    p=0.1383  normal=True
  score     p=0.8387  normal=True


In [17]:
# Categorical analysis
cat_exp = explore_data(df.select(['dept']))
cat     = cat_exp.analyze_categorical_variables()['categorical_analysis']
print('Categorical columns found:', list(cat.keys()))

📋 Categorical Variables Analysis

Warning: Could not analyze categorical variable dept: conversion from `str` to `f64` failed in column 'dept' for 2 
out of 2 values: ["eng", "hr"]

Categorical columns found: ['dept']


In [18]:
# Terminal plots
print('--- Histogram: score ---')
explorer.plot_histogram('score')

print('\n--- Scatter: age vs salary ---')
explorer.plot_scatter('age', 'salary')

print('\n--- Correlation heatmap ---')
explorer.plot_correlation_heatmap()

--- Histogram: score ---


📊 Histogram: score

                                Distribution of score                           
    ┌──────────────────────────────────────────────────────────────────────────┐
1.00┤████   ████       ████       ████        ████       ████       ████   ████│
    │████   ████       ████       ████        ████       ████       ████   ████│
0.83┤████   ████       ████       ████        ████       ████       ████   ████│
    │████   ████       ████       ████        ████       ████       ████   ████│
    │████   ████       ████       ████        ████       ████       ████   ████│
0.67┤████   ████       ████       ████        ████       ████       ████   ████│
    │████   ████       ████       ████        ████       ████       ████   ████│
0.50┤████   ████       ████       ████        ████       ████       ████   ████│
    │████   ████       ████       ████        ████       ████       ████   ████│
0.33┤████   ████       ████       ████        ████       ████       ████   ████│
    │████   ████       ████ 

📈 Statistics for score:
• Count: 8
• Mean: 86.950
• Median: 86.900
• Std Dev: 7.146
• Min: 76.300
• Max: 97.500


--- Scatter: age vs salary ---


...

📈 Correlation: 0.9843

💡 Interpretation: Very Strong positive correlation


--- Correlation heatmap ---


🔥 Correlation Heatmap

┏━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┓
┃ Column       ┃ age      ┃ salary   ┃ score    ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━┩
│ age          │ 1.00     │ 0.98     │ 0.75     │
│ salary       │ 0.98     │ 1.00     │ 0.80     │
│ score        │ 0.75     │ 0.80     │ 1.00     │
└──────────────┴──────────┴──────────┴──────────┘

## 5. Cleaner

In [19]:
# MEAN imputation for numerics
df_mean = (
    DataCleaner(df)
    .handle_missing_values(strategy=ImputationStrategy.MEAN, columns=['age', 'salary'])
    .get_cleaned_data()
)
print(f'After MEAN fill — age nulls: {df_mean["age"].null_count()}  salary nulls: {df_mean["salary"].null_count()}')

🔧 Handling Missing Values (Mean)

✅ Missing values reduced from 3 to 1

After MEAN fill — age nulls: 0  salary nulls: 0


In [20]:
# MODE imputation for categoricals
df_mode = (
    DataCleaner(df)
    .handle_missing_values(strategy=ImputationStrategy.MODE, columns=['name'])
    .get_cleaned_data()
)
print(f'After MODE fill — name nulls: {df_mode["name"].null_count()}')

🔧 Handling Missing Values (Mode)

✅ Missing values reduced from 3 to 2

After MODE fill — name nulls: 0


In [21]:
# Custom value
df_cust = (
    DataCleaner(df)
    .handle_missing_values(strategy=ImputationStrategy.CUSTOM_VALUE,
                           columns=['name'], custom_value='Unknown')
    .get_cleaned_data()
)
print(f'Custom fill: Unknown in name? {"Unknown" in df_cust["name"].to_list()}')

🔧 Handling Missing Values (Custom)

✅ Missing values reduced from 3 to 2

Custom fill: Unknown in name? True


In [22]:
# Forward fill
df_ff = (
    DataCleaner(df)
    .handle_missing_values(strategy=ImputationStrategy.FORWARD_FILL)
    .get_cleaned_data()
)
print(f'Forward fill — total nulls remaining: {df_ff.null_count().sum_horizontal().sum()}')

🔧 Handling Missing Values (Ffill)

✅ Missing values reduced from 3 to 0

Forward fill — total nulls remaining: 0


In [23]:
# Remove duplicates
df_dups = pl.concat([df, df.head(3)])
df_dedup = DataCleaner(df_dups).remove_duplicates().get_cleaned_data()
print(f'Dedup: {len(df_dups)} → {len(df_dedup)} rows')

🔧 Removing Duplicate Rows

✅ Removed 3 duplicate rows (10 remaining)

Dedup: 13 → 10 rows


In [24]:
# Outlier handling
df_clean = (
    DataCleaner(df)
    .handle_missing_values(strategy=ImputationStrategy.MEAN, columns=['age', 'salary'])
    .handle_missing_values(strategy=ImputationStrategy.MODE, columns=['name'])
    .get_cleaned_data()
)

df_no_outliers = DataCleaner(df_clean).handle_outliers(method='iqr', action='remove').get_cleaned_data()
df_capped      = DataCleaner(df_clean).handle_outliers(method='iqr', action='cap').get_cleaned_data()
print(f'Original: {len(df_clean)} rows')
print(f'Remove:   {len(df_no_outliers)} rows')
print(f'Cap:      {len(df_capped)} rows  (same as original)')

🔧 Handling Missing Values (Mean)

✅ Missing values reduced from 3 to 1

🔧 Handling Missing Values (Mode)

✅ Missing values reduced from 1 to 0

🎯 Handling Outliers (IQR, remove)

                 Outlier Handling Summary (IQR)                 
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column          ┃ Outliers Found ┃ Action Taken              ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ id              │ 0              │ none                      │
│ age             │ 1              │ removed 1 rows            │
│ salary          │ 1              │ removed 1 rows            │
│ score           │ 0              │ none                      │
└─────────────────┴────────────────┴───────────────────────────┘

🎯 Handling Outliers (IQR, cap)

                 Outlier Handling Summary (IQR)                 
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column          ┃ Outliers Found ┃ Action Taken              ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ id              │ 0              │ none                      │
│ age             │ 1              │ capped 1 values           │
│ salary          │ 1              │ capped 1 values           │
│ score           │ 0              │ none                      │
└─────────────────┴────────────────┴───────────────────────────┘

Original: 10 rows
Remove:   8 rows
Cap:      10 rows  (same as original)


In [25]:
# Text cleaning
df_txt = DataCleaner(df_clean).clean_text_data(columns=['name', 'dept']).get_cleaned_data()
print('Text cleaned — dept values:', df_txt['dept'].unique().sort().to_list())

# Quick clean (all-in-one for ML)
df_qc = quick_clean(df)
print(f'Quick clean: {df_qc.shape}')

# Full chained pipeline
df_chained = (
    DataCleaner(df)
    .handle_missing_values(strategy=ImputationStrategy.MEAN,  columns=['age', 'salary'])
    .handle_missing_values(strategy=ImputationStrategy.MODE,  columns=['name'])
    .remove_duplicates()
    .get_cleaned_data()
)
print(f'Chained: {df_chained.shape}  nulls={df_chained.null_count().sum_horizontal().sum()}')

📝 Cleaning Text Data (2 columns)

✅ Text cleaning complete for 2 columns

Text cleaned — dept values: ['eng', 'hr']


✅ Registered cleaning strategy: 'basic_cleaning'

🧹 Applying strategy: basic_cleaning

...

🔧 Handling Missing Values (Mode)

✅ Missing values reduced from 1 to 0

🔧 Removing Duplicate Rows

✅ Removed 0 duplicate rows (10 remaining)

Chained: (10, 7)  nulls=0


## 6. Transformer

In [26]:
df_base = df_chained.drop_nulls()
print(f'Base for transform: {df_base.shape}')
df_base

Base for transform: (10, 7)


id,name,age,salary,dept,score,active
i64,str,f64,f64,str,f64,bool
1,"""Alice""",25.0,55000.0,"""eng""",88.5,true
2,"""Bob""",30.0,72000.0,"""eng""",92.0,true
3,"""Carol""",22.0,48000.0,"""hr""",76.3,true
4,"""Dave""",35.0,95000.0,"""eng""",95.0,true
5,"""Eve""",28.0,61000.0,"""hr""",82.1,false
6,"""Frank""",45.0,120000.0,"""eng""",97.5,true
7,"""Grace""",31.0,68000.0,"""hr""",78.9,true
8,"""Eve""",30.222222,53000.0,"""hr""",65.0,false
9,"""Henry""",27.0,59000.0,"""eng""",85.3,true


In [27]:
# Label encoding
t = DataTransformer(df_base)
t.encode_categorical_features(method=EncodingMethod.LABEL, columns=['dept'])
enc = t.get_transformed_data()
print(f'Label encode dept — unique values: {enc["dept"].unique().sort().to_list()}')

🏷️ Encoding Categorical Features (Label)

                                     Encoding Results (Label)                                     
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column          ┃ Status     ┃ New Columns  ┃ Unique Values ┃ Details                          ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ dept            │ ✅ Success │ 0            │ 2             │ Method: label, Mapping available │
└─────────────────┴────────────┴──────────────┴───────────────┴──────────────────────────────────┘

✅ Categorical encoding complete (0 new columns created)

Label encode dept — unique values: ['0', '1']


In [28]:
# One-hot encoding
t_oh = DataTransformer(df_base)
t_oh.encode_categorical_features(method=EncodingMethod.ONEHOT, columns=['dept'])
oh = t_oh.get_transformed_data()
print(f'One-hot: {df_base.shape[1]} → {oh.shape[1]} columns')
print('New columns:', [c for c in oh.columns if c not in df_base.columns])

🏷️ Encoding Categorical Features (Onehot)

                              Encoding Results (Onehot)                               
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column          ┃ Status     ┃ New Columns  ┃ Unique Values ┃ Details              ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ dept            │ ✅ Success │ 2            │ 2             │ Method: onehot       │
└─────────────────┴────────────┴──────────────┴───────────────┴──────────────────────┘

✅ Categorical encoding complete (2 new columns created)

One-hot: 7 → 8 columns
New columns: ['dept_eng', 'dept_hr']


In [29]:
# Standard scaling
t_std = DataTransformer(df_base)
t_std.encode_categorical_features(method=EncodingMethod.LABEL)
t_std.scale_features(method=ScalingMethod.STANDARD, columns=['age', 'salary', 'score'])
scaled = t_std.get_transformed_data()
print(f'Standard scaled — age mean={scaled["age"].mean():.6f}  (≈0)')

🏷️ Encoding Categorical Features (Label)

                                     Encoding Results (Label)                                     
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column          ┃ Status     ┃ New Columns  ┃ Unique Values ┃ Details                          ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ name            │ ✅ Success │ 0            │ 9             │ Method: label                    │
│ dept            │ ✅ Success │ 0            │ 2             │ Method: label, Mapping available │
└─────────────────┴────────────┴──────────────┴───────────────┴──────────────────────────────────┘

✅ Categorical encoding complete (0 new columns created)

⚖️ Scaling Features (Standard)

                           Scaling Results (Standard)                           
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Column          ┃ Original Mean ┃ Original Std ┃ New Mean     ┃ New Std      ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ age             │ 30.222        │ 6.268        │ 0.000        │ 1.054        │
│ salary          │ 70111.111     │ 21941.561    │ 0.000        │ 1.054        │
│ score           │ 85.060        │ 9.797        │ 0.000        │ 1.054        │
└─────────────────┴───────────────┴──────────────┴──────────────┴──────────────┘

✅ Feature scaling complete (3 columns scaled)

Standard scaled — age mean=0.000000  (≈0)


In [30]:
# MinMax scaling
t_mm = DataTransformer(df_base)
t_mm.encode_categorical_features(method=EncodingMethod.LABEL)
t_mm.scale_features(method=ScalingMethod.MINMAX, columns=['age', 'salary', 'score'])
mm = t_mm.get_transformed_data()
print(f'MinMax score: [{mm["score"].min():.2f}, {mm["score"].max():.2f}]  (should be [0, 1])')

🏷️ Encoding Categorical Features (Label)

                                     Encoding Results (Label)                                     
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column          ┃ Status     ┃ New Columns  ┃ Unique Values ┃ Details                          ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ name            │ ✅ Success │ 0            │ 9             │ Method: label                    │
│ dept            │ ✅ Success │ 0            │ 2             │ Method: label, Mapping available │
└─────────────────┴────────────┴──────────────┴───────────────┴──────────────────────────────────┘

✅ Categorical encoding complete (0 new columns created)

⚖️ Scaling Features (Minmax)

                            Scaling Results (Minmax)                            
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Column          ┃ Original Mean ┃ Original Std ┃ New Mean     ┃ New Std      ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ age             │ 30.222        │ 6.268        │ 0.357        │ 0.273        │
│ salary          │ 70111.111     │ 21941.561    │ 0.307        │ 0.305        │
│ score           │ 85.060        │ 9.797        │ 0.617        │ 0.301        │
└─────────────────┴───────────────┴──────────────┴──────────────┴──────────────┘

✅ Feature scaling complete (3 columns scaled)

MinMax score: [0.00, 1.00]  (should be [0, 1])


In [31]:
# Full chained transform pipeline
final = (
    DataTransformer(df_base)
    .encode_categorical_features(method=EncodingMethod.LABEL)
    .scale_features(method=ScalingMethod.ROBUST)
    .get_transformed_data()
)
print(f'Final transformed shape: {final.shape}')
final

🏷️ Encoding Categorical Features (Label)

                                     Encoding Results (Label)                                     
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column          ┃ Status     ┃ New Columns  ┃ Unique Values ┃ Details                          ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ name            │ ✅ Success │ 0            │ 9             │ Method: label                    │
│ dept            │ ✅ Success │ 0            │ 2             │ Method: label, Mapping available │
└─────────────────┴────────────┴──────────────┴───────────────┴──────────────────────────────────┘

✅ Categorical encoding complete (0 new columns created)

⚖️ Scaling Features (Robust)

                            Scaling Results (Robust)                            
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Column          ┃ Original Mean ┃ Original Std ┃ New Mean     ┃ New Std      ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ id              │ 5.500         │ 3.028        │ 0.000        │ 0.673        │
│ age             │ 30.222        │ 6.268        │ 0.203        │ 1.763        │
│ salary          │ 70111.111     │ 21941.561    │ 0.361        │ 1.413        │
│ score           │ 85.060        │ 9.797        │ -0.156       │ 0.830        │
└─────────────────┴───────────────┴──────────────┴──────────────┴──────────────┘

✅ Feature scaling complete (4 columns scaled)

Final transformed shape: (10, 7)


id,name,age,salary,dept,score,active
f64,str,f64,f64,str,f64,bool
-1.0,"""0""",-1.265625,-0.611807,"""0""",0.135593,true
-0.777778,"""1""",0.140625,0.483005,"""0""",0.432203,true
-0.555556,"""2""",-2.109375,-1.062612,"""1""",-0.898305,true
-0.333333,"""3""",1.546875,1.964222,"""0""",0.686441,true
-0.111111,"""4""",-0.421875,-0.225403,"""1""",-0.40678,false
0.111111,"""5""",4.359375,3.57424,"""0""",0.898305,true
0.333333,"""6""",0.421875,0.225403,"""1""",-0.677966,true
0.555556,"""4""",0.203125,-0.740608,"""1""",-1.855932,false
0.777778,"""7""",-0.703125,-0.354204,"""0""",-0.135593,true


## 7. Validator

In [32]:
# Infer schema from the clean dataset
schema = infer_schema(df_clean)
print(f'Inferred {len(schema.columns)} column schemas:')
for cs in schema.columns:
    print(f'  {cs.name:10}  dtype={cs.dtype:12}  nullable={cs.nullable}  max_missing={cs.max_missing_pct:.1f}%')

Inferred 7 column schemas:
  id          dtype=Int64         nullable=False  max_missing=5.0%
  name        dtype=String        nullable=False  max_missing=5.0%
  age         dtype=Float64       nullable=False  max_missing=5.0%
  salary      dtype=Float64       nullable=False  max_missing=5.0%
  dept        dtype=String        nullable=False  max_missing=5.0%
  score       dtype=Float64       nullable=False  max_missing=5.0%
  active      dtype=Boolean       nullable=False  max_missing=5.0%


In [33]:
# Save schema and reload
schema_path = tmp / 'schema.json'
schema.to_json(schema_path)
reloaded = DataSchema.from_json(schema_path)
print(f'Schema saved to {schema_path}')
print(f'Reloaded {len(reloaded.columns)} columns — roundtrip OK')

Schema saved to /tmp/wrang_notebook_test/schema.json
Reloaded 7 columns — roundtrip OK


In [34]:
# Validate clean df against its own inferred schema — should PASS
result = DataValidator(schema).validate(df_clean)
result.display()
print(f'\nPassed: {result.passed}  Errors: {len(result.errors)}  Warnings: {len(result.warnings)}')

✅  All validation checks passed!


Passed: True  Errors: 0  Warnings: 0


In [35]:
# Strict schema — will produce violations
strict = DataSchema(
    columns=[
        ColumnSchema(name='id',     dtype='Int64',   nullable=False, unique=True),
        ColumnSchema(name='age',    dtype='Int64',   nullable=False, min_value=18, max_value=65),
        ColumnSchema(name='salary', dtype='Float64', nullable=False, min_value=0.0, max_value=80000.0),
        ColumnSchema(name='dept',   dtype='String',  nullable=False,
                     allowed_values=['eng', 'hr', 'finance']),
        ColumnSchema(name='score',  dtype='Float64', nullable=False, min_value=0.0, max_value=100.0),
    ],
    require_all_columns=False,
    allow_extra_columns=True,
)

result2 = validate_data(df_clean, strict)
result2.display()
print(f'\nPassed: {result2.passed}  Errors: {len(result2.errors)}  Warnings: {len(result2.warnings)}')

Validation Result: FAILED

Errors: 2  Warnings: 0

 Column   Check       Severity   Message                                     Actual     Expected  
 ───────────────────────────────────────────────────────────────────────────────────────────────── 
  age      dtype        ERROR     Dtype mismatch.                             Float64    Int64     
  salary   max_value    ERROR     Maximum value 120000.0 exceeds threshold.   120000.0   ≤80000.0 


Passed: False  Errors: 2  Warnings: 0


In [36]:
# Programmatic access to violations
print('All violations:')
for v in result2.violations:
    print(f'  [{v.severity.upper():7}] {v.column:10} / {v.check:20} | {v.message}')

# to_dict() for downstream use (CI, logging, etc.)
summary = result2.to_dict()
print(f'\nJSON summary: {json.dumps({k: v for k, v in summary.items() if k != "violations"}, indent=2)}')

All violations:
  [ERROR  ] age        / dtype                | Dtype mismatch.
  [ERROR  ] salary     / max_value            | Maximum value 120000.0 exceeds threshold.

JSON summary: {
  "passed": false,
  "error_count": 2,
  "warning_count": 0
}


## 8. DuckDB SQL

In [37]:
try:
    import duckdb

    con = duckdb.connect(':memory:')
    con.register('data', df_clean.to_arrow())

    # Basic SELECT
    q1 = con.execute('SELECT * FROM data LIMIT 5').pl()
    print('SELECT * LIMIT 5:')
    display(q1)

    # Aggregation
    q2 = con.execute('SELECT dept, AVG(salary) as avg_sal, COUNT(*) as n FROM data GROUP BY dept').pl()
    print('\nAverage salary by department:')
    display(q2)

    # Filter
    q3 = con.execute('SELECT name, age, score FROM data WHERE score > 85 ORDER BY score DESC').pl()
    print('\nHigh scorers (>85):')
    display(q3)

except ImportError:
    print('duckdb not installed — skipping. Install with: pip install duckdb')

SELECT * LIMIT 5:


id,name,age,salary,dept,score,active
i64,str,f64,f64,str,f64,bool
1,"""Alice""",25.0,55000.0,"""eng""",88.5,true
2,"""Bob""",30.0,72000.0,"""eng""",92.0,true
3,"""Carol""",22.0,48000.0,"""hr""",76.3,true
4,"""Dave""",35.0,95000.0,"""eng""",95.0,true
5,"""Eve""",28.0,61000.0,"""hr""",82.1,false



Average salary by department:


dept,avg_sal,n
str,f64,i64
"""eng""",80200.0,5
"""hr""",60022.222222,5



High scorers (>85):


name,age,score
str,f64,f64
"""Frank""",45.0,97.5
"""Dave""",35.0,95.0
"""Bob""",30.0,92.0
"""Iris""",29.0,90.0
"""Alice""",25.0,88.5
"""Henry""",27.0,85.3


## 9. Real Dataset — Titanic

In [38]:
titanic_path = Path('datasets/titanic.csv')

if not titanic_path.exists():
    print('titanic.csv not found in datasets/ — skipping')
else:
    # Load
    t = load_data(titanic_path)
    print(f'Titanic: {t.shape}')

    # Inspect
    ti = inspect_data(t)
    info = ti.get_basic_info()
    print(f'Missing cells: {info["missing_values_total"]}')
    ti.display_data_quality()

    # Clean
    t_clean = (
        DataCleaner(t)
        .handle_missing_values(strategy=ImputationStrategy.MEDIAN,       columns=['Age'])
        .handle_missing_values(strategy=ImputationStrategy.MODE,         columns=['Embarked'])
        .handle_missing_values(strategy=ImputationStrategy.CUSTOM_VALUE, columns=['Cabin'], custom_value='Unknown')
        .remove_duplicates()
        .get_cleaned_data()
    )
    print(f'Cleaned: {t_clean.shape}  nulls={t_clean.null_count().sum_horizontal().sum()}')

    # Schema validation
    t_schema = infer_schema(t_clean)
    t_result = DataValidator(t_schema).validate(t_clean)
    print(f'Schema validation: {"PASS" if t_result.passed else "FAIL"}')

✅ Successfully loaded 891 rows × 12 columns

Titanic: (891, 12)
Missing cells: 866


╭──── 🚫 Missing Values ─────╮            ╭────── 🔧 Data Types ───────╮        ╭────── 🔄 Duplicates ───────╮     
│ Missing Values Summary     │            │ Data Types Distribution    │        │ Duplicate Analysis         │     
│                            │            │                            │        │                            │     
│ • Columns with missing: 3  │            │ • Numeric: 7               │        │ • Duplicate rows: 0        │     
│ • Total missing cells: 866 │            │ • Categorical: 5           │        │ • Duplicate %: 0.0%        │     
│ • Overall missing %: 8.1%  │            │ • DateTime: 0              │        │ • Unique rows: 891         │     
│                            │            │                            │        │                            │     
│ Worst Columns:             │            │ Type Details:              │        │ Status:                    │     
│ • Cabin: 77.1%             │            │ • Int64: 5                 │        │ ✅ No duplicates found     │     
│ • Age: 19.9%               │            │ • String: 5                │        ╰────────────────────────────╯     
│ • Embarked: 0.2%           │            │ • Float64: 2               │                                           
╰────────────────────────────╯            ╰────────────────────────────╯

🔧 Handling Missing Values (Median)

✅ Missing values reduced from 866 to 689

...

🔧 Handling Missing Values (Custom)

✅ Missing values reduced from 687 to 0

🔧 Removing Duplicate Rows

✅ Removed 0 duplicate rows (891 remaining)

Cleaned: (891, 12)  nulls=0
Schema validation: PASS


## 10. Config

In [39]:
config = get_config()
print('Current config:')
print(f'  random_state:      {config.random_state}')
print(f'  chunk_size:        {config.chunk_size}')
print(f'  outlier_method:    {config.outlier_method}')
print(f'  missing_threshold: {config.missing_value_threshold}')

updated = update_config(random_state=42, chunk_size=2000)
print(f'\nUpdated: random_state={updated.random_state}  chunk_size={updated.chunk_size}')

reset_config()
print(f'After reset: random_state={get_config().random_state}')

Current config:
  random_state:      42
  chunk_size:        10000
  outlier_method:    iqr
  missing_threshold: 0.9

Updated: random_state=42  chunk_size=2000
After reset: random_state=42


## ✅ All cells passed

If you reached here with no errors, the `ride` package is working correctly in a notebook environment.

**Test artifacts saved to:** `/tmp/ride_notebook_test/`